# 05B Global Model Transfer Only

This notebook runs a strict saved-artifact transfer evaluation on M5 using synthetic-trained global boosting artifacts.

Important:
- this is an honest saved-model transfer test
- it does **not** retrain on M5
- it produces transfer metrics on monthly aggregated M5 data
- it is not the same as a Kaggle daily submission, because the saved synthetic artifacts are monthly models. This notebook is transfer evaluation only and does not generate uploadable Kaggle CSVs.
- prerequisite: saved artifacts must already exist under `modeling/outputs/artifacts` (for example from notebook 02 global model training and artifact save)

In [1]:
from pathlib import Path
import pandas as pd

M5_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/external-data/m5-forecasting-accuracy')
REPORTS_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports')
SCRIPT = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/scripts/m5_saved_artifact_transfer.py')
TAG = 'portable_m5_transfer_auto_capped'

def resolve_report_path(kind: str, preferred_tag: str = TAG) -> Path:
    if not REPORTS_DIR.exists():
        raise FileNotFoundError(
            f'Reports directory not found: {REPORTS_DIR}. Run the transfer cell above first.'
        )

    preferred = REPORTS_DIR / f'{preferred_tag}_{kind}.csv'
    if preferred.exists():
        return preferred

    available = sorted(p.name for p in REPORTS_DIR.glob(f'{preferred_tag}_*.csv'))
    raise FileNotFoundError(
        f'Missing transfer report: {preferred.name}. Run the transfer cell above first. '
        f'Available transfer CSVs for tag {preferred_tag!r}: {available}'
    )


## Run Transfer Evaluation

In [2]:
ARTIFACTS_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/artifacts')
required_checks = [
    ARTIFACTS_DIR / 'P' / 'xgboost_h1' / 'production' / 'metadata.json',
]
missing = [str(p) for p in required_checks if not p.exists()]

if missing:
    raise FileNotFoundError(
        'Missing saved model artifacts required for transfer evaluation. '
        'Run notebook 02_global_model_training_and_artifact_save.ipynb first. '
        f'Missing examples: {missing}'
    )

!python "{SCRIPT}" --m5-dir "{M5_DIR}" --granularity dept_store --datasets P --models XGBOOST --tag "{TAG}" --calibration recent_level_auto_capped --calibration-max-weight 0.8


Saved predictions: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_auto_capped_predictions.csv
Saved metrics: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_auto_capped_metrics.csv
Saved summary: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_auto_capped_summary.csv
Saved calibration: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_auto_capped_calibration.csv


## Summary

In [3]:
!python "/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/scripts/m5_saved_artifact_transfer.py" \
  --m5-dir "/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/external-data/m5-forecasting-accuracy" \
  --granularity dept_store \
  --datasets P \
  --models XGBOOST \
  --tag "portable_m5_transfer_auto_capped" \
  --calibration recent_level_auto_capped --calibration-max-weight 0.8


Saved predictions: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_auto_capped_predictions.csv
Saved metrics: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_auto_capped_metrics.csv
Saved summary: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_auto_capped_summary.csv
Saved calibration: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_auto_capped_calibration.csv


In [4]:
summary_path = resolve_report_path('summary')
pd.read_csv(summary_path)


,dataset,model,split,horizon,n_obs,WAPE,RMSE,Bias,MASE_mean,wQL50,under_forecast_rate
0,M5_TRANSFER_P,XGBOOST_recent_level_auto_capped,test,0,10080,0.131981,3833.020121,-1091.988664,1.171654,1117.384557,0.680655


## Detailed Horizon Metrics

In [5]:
metrics_path = resolve_report_path('metrics')
pd.read_csv(metrics_path).head(50)

,dataset,model,split,horizon,n_obs,WAPE,RMSE,Bias,MASE_mean,wQL50,under_forecast_rate
0,M5_TRANSFER_P,XGBOOST_recent_level_auto_capped,test,1,840,0.097779,2987.245332,-541.060592,0.817672,827.824909,0.598810
1,M5_TRANSFER_P,XGBOOST_recent_level_auto_capped,test,2,840,0.106817,3292.889867,-738.712907,0.931934,904.339966,0.592857
2,M5_TRANSFER_P,XGBOOST_recent_level_auto_capped,test,3,840,0.113293,3501.966231,-843.925354,1.009495,959.166102,0.598810
3,M5_TRANSFER_P,XGBOOST_recent_level_auto_capped,test,4,840,0.123894,3685.726056,-617.596748,1.077425,1048.921592,0.651190
4,M5_TRANSFER_P,XGBOOST_recent_level_auto_capped,test,5,840,0.124125,3670.951749,-806.757463,1.120743,1050.871218,0.633333
5,M5_TRANSFER_P,XGBOOST_recent_level_auto_capped,test,6,840,0.132086,3865.220910,-1046.229479,1.149652,1118.276529,0.667857
6,M5_TRANSFER_P,XGBOOST_recent_level_auto_capped,test,7,840,0.141238,4146.692763,-909.193543,1.232813,1195.759009,0.647619
7,M5_TRANSFER_P,XGBOOST_recent_level_auto_capped,test,8,840,0.139652,3990.459533,-1167.142759,1.266092,1182.326035,0.698810
8,M5_TRANSFER_P,XGBOOST_recent_level_auto_capped,test,9,840,0.146097,4085.636937,-1691.659510,1.313961,1236.898116,0.757143
9,M5_TRANSFER_P,XGBOOST_recent_level_auto_capped,test,10,840,0.142293,4117.274038,-1340.807700,1.294548,1204.690200,0.748810


## Why This Is Not a Kaggle Submission

The saved artifacts are monthly synthetic-trained models. Kaggle M5 submission requires 28-day daily item-store forecasts. That means:
- this notebook is valid for transfer evaluation
- it is not valid for strict Kaggle submission generation from the same saved monthly artifacts